# Diagnostic: how much do our existing models actually disagree?

No training. This reads the eight out-of-fold vectors already saved in
`artifacts/oof/` and answers two questions that the CatBoost run cannot answer on
its own.

**1. What does "high correlation" mean on this dataset?** `06_catboost.ipynb`
contains hard thresholds: below 0.97 Spearman the blending plan is alive, above 0.99
the two models are the same thing wearing different hats. Those numbers were written
from general experience, not from this data. Eight LightGBM variants that differ only
in tree count and learning rate give a reference scale. If two runs that differ only
in learning rate sit at 0.998, then 0.99 is not a demanding bar and the threshold is
wrong.

**2. What is the floor on blending gain?** Rank-averaging near-identical models is
the least promising ensemble available. Whatever it yields is the number a genuinely
diverse model has to beat to be worth its runtime.

Cost is a few seconds and no ledger row is written, so this is in the same category as
notebooks 04 and 05: a diagnostic that changes the plan without producing a score.

**Caveat carried in from `NOTES.md`:** experiments 1 to 5 predate the determinism
flags and carry roughly 1e-4 of run-to-run noise. That is far too small to disturb a
correlation measured on 691,369 rows, but any blend AUC involving them is quoted with
that noise attached. The three deterministic-era runs (6, 7, 8) are the clean set and
the headline numbers come from those.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5
TARGET = "addicted_label"
ID = "id"

pd.set_option("display.width", 140)
print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 2.5.1 | pandas 3.0.5


In [2]:
def locate():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw" / "train.csv").exists():
            return base, base / "data" / "raw"
    kag = Path("/kaggle/input/playground-series-s6e8")
    if (kag / "train.csv").exists():
        return Path("/kaggle/working"), kag
    raise FileNotFoundError("could not find train.csv")


REPO, RAW = locate()
OOF_DIR = REPO / "artifacts" / "oof"

# Only the target column is needed. Nothing here touches the features.
train = pd.read_csv(RAW / "train.csv", usecols=[ID, TARGET])
y = train[TARGET].to_numpy()

# Identical construction to notebooks 01 to 03, so fold-mean AUCs are comparable
# to the ledger rather than merely similar to them.
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = np.full(len(train), -1, dtype=int)
for i, (_, va) in enumerate(skf.split(train, y)):
    folds[va] = i
assert (folds >= 0).all()

print(f"{len(train):,} rows, target rate {y.mean():.6f}")

691,369 rows, target rate 0.709424


## Load the OOF vectors and reconcile them against the ledger

Before any of these are compared, each one has to prove it is the vector its filename
claims. Recomputing the fold-mean AUC and checking it against `experiments.csv` is the
only provenance available under the notebook layout, since a ledger row here carries no
config hash and no git SHA.

Discrepancies are printed rather than asserted. `nbconvert` discards all output from a
run in which a cell raises, so a hard assertion here would destroy the evidence needed
to diagnose it.

In [3]:
# filename -> (ledger id, ledger cv_mean, deterministic era?)
REGISTRY = [
    ("lgbm_trees100_seed42.npy", 2, 0.954947, False),
    ("lgbm_trees300_seed42.npy", 3, 0.960605, False),
    ("lgbm_trees1000_seed42.npy", 4, 0.962141, False),
    ("lgbm_trees2000_seed42.npy", 5, 0.961832, False),
    ("lgbm_lr01_n1000_seed42.npy", 6, 0.962198, True),
    ("lgbm_lr005_n2000_seed42.npy", 7, 0.963210, True),
    ("lgbm_lr003_n3333_seed42.npy", 8, 0.963275, True),
]


def fold_mean_auc(p):
    return float(np.mean([roc_auc_score(y[folds == f], p[folds == f])
                          for f in range(N_SPLITS)]))


def fold_sd_auc(p):
    return float(np.std([roc_auc_score(y[folds == f], p[folds == f])
                         for f in range(N_SPLITS)]))


oofs, meta = {}, []
for fname, exp_id, ledger_cv, det in REGISTRY:
    path = OOF_DIR / fname
    if not path.exists():
        print(f"MISSING {fname}")
        continue
    p = np.load(path)
    if len(p) != len(y):
        print(f"LENGTH MISMATCH {fname}: {len(p)} vs {len(y)}")
        continue
    key = f"exp{exp_id}"
    oofs[key] = p
    fm, pooled = fold_mean_auc(p), float(roc_auc_score(y, p))
    meta.append({"key": key, "file": fname.replace("_seed42.npy", ""),
                 "det": det, "ledger_cv": ledger_cv, "fold_mean": fm,
                 "delta": fm - ledger_cv, "pooled": pooled,
                 "fold_sd": fold_sd_auc(p)})

recon = pd.DataFrame(meta)
print(recon.to_string(index=False,
                      formatters={"ledger_cv": "{:.6f}".format,
                                  "fold_mean": "{:.6f}".format,
                                  "delta": "{:+.2e}".format,
                                  "pooled": "{:.6f}".format,
                                  "fold_sd": "{:.6f}".format}))

worst = recon["delta"].abs().max()
print(f"\nlargest disagreement with the ledger: {worst:.2e}")
print("every saved vector reproduces its ledger row" if worst < 1e-6
      else "AT LEAST ONE VECTOR DOES NOT MATCH ITS LEDGER ROW, investigate before using it")
print("\nNote the pooled column sits below fold_mean throughout. AUC does not decompose")
print("across folds, so pooled and fold-mean are different statistics. Compare like")
print("with like: the ledger is fold-mean, and 06_catboost.ipynb currently reads pooled.")

 key             file   det ledger_cv fold_mean     delta   pooled  fold_sd
exp2    lgbm_trees100 False  0.954947  0.954947 -2.99e-07 0.954944 0.000645
exp3    lgbm_trees300 False  0.960605  0.960605 -1.58e-07 0.960604 0.000688
exp4   lgbm_trees1000 False  0.962141  0.962141 -9.49e-08 0.962137 0.000859
exp5   lgbm_trees2000 False  0.961832  0.961832 +3.24e-07 0.961807 0.000952
exp6  lgbm_lr01_n1000  True  0.962198  0.962198 +2.37e-07 0.962195 0.000816
exp7 lgbm_lr005_n2000  True  0.963210  0.963210 -1.73e-07 0.963209 0.000591
exp8 lgbm_lr003_n3333  True  0.963275  0.963275 -4.61e-07 0.963274 0.000549

largest disagreement with the ledger: 4.61e-07
every saved vector reproduces its ledger row

Note the pooled column sits below fold_mean throughout. AUC does not decompose
across folds, so pooled and fold-mean are different statistics. Compare like
with like: the ledger is fold-mean, and 06_catboost.ipynb currently reads pooled.


## Question 1: the correlation scale

Spearman is computed as Pearson on percentile ranks, which is the same statistic and
avoids ranking the same 691,369-row vector twenty-one times.

In [4]:
keys = list(oofs)
ranks = {k: pd.Series(v).rank(pct=True).to_numpy() for k, v in oofs.items()}
R = np.corrcoef(np.vstack([ranks[k] for k in keys]))
spearman = pd.DataFrame(R, index=keys, columns=keys)

print("Spearman rank correlation between out-of-fold predictions\n")
print(spearman.to_string(float_format="{:.4f}".format))

det_keys = [m["key"] for m in meta if m["det"]]
off = spearman.where(~np.eye(len(keys), dtype=bool))
det_off = spearman.loc[det_keys, det_keys].where(~np.eye(len(det_keys), dtype=bool))

print(f"\nall pairs          : min {off.min().min():.4f}  max {off.max().max():.4f}")
print(f"deterministic three: min {det_off.min().min():.4f}  "
      f"max {det_off.max().max():.4f}")
print("\nThe first line is the widest spread one model family produces here, from an")
print("underfit 100-tree run to a tuned one. The second is the spread among runs that")
print("differ only in learning rate. Both are within-family, so both are the scale")
print("against which a different family has to look different.")

Spearman rank correlation between out-of-fold predictions

       exp2   exp3   exp4   exp5   exp6   exp7   exp8
exp2 1.0000 0.9930 0.9840 0.9741 0.9842 0.9890 0.9889
exp3 0.9930 1.0000 0.9891 0.9787 0.9898 0.9930 0.9927
exp4 0.9840 0.9891 1.0000 0.9848 0.9943 0.9885 0.9877
exp5 0.9741 0.9787 0.9848 1.0000 0.9834 0.9790 0.9783
exp6 0.9842 0.9898 0.9943 0.9834 1.0000 0.9892 0.9882
exp7 0.9890 0.9930 0.9885 0.9790 0.9892 1.0000 0.9981
exp8 0.9889 0.9927 0.9877 0.9783 0.9882 0.9981 1.0000

all pairs          : min 0.9741  max 0.9981
deterministic three: min 0.9882  max 0.9981

The first line is the widest spread one model family produces here, from an
underfit 100-tree run to a tuned one. The second is the spread among runs that
differ only in learning rate. Both are within-family, so both are the scale
against which a different family has to look different.


In [5]:
within = float(det_off.min().min())
widest = float(off.min().min())
print(f"06_catboost.ipynb thresholds: alive below 0.97, dead above 0.99\n")
print(f"tightest within-family pair (lr only) : {det_off.max().max():.4f}")
print(f"loosest within-family pair (lr only)  : {within:.4f}")
print(f"loosest pair of any kind (capacity)   : {widest:.4f}\n")

if widest < 0.97:
    print("VERDICT ON THE THRESHOLDS: too lenient. Two runs of the SAME model family")
    print("already clear the 0.97 'genuine disagreement' bar, so CatBoost clearing it")
    print("would prove nothing. Raise the bar to the within-family floor above before")
    print("reading CatBoost's number, or the diversity test cannot fail.")
elif widest < 0.99:
    print("VERDICT ON THE THRESHOLDS: the 0.99 'same model' bar is unsafe. Same-family")
    print("runs already sit below it, so it cannot separate families. The 0.97 bar")
    print("still discriminates. Treat 0.99 as uninformative rather than as a kill.")
else:
    print("VERDICT ON THE THRESHOLDS: they survive. Every within-family pair sits above")
    print("0.99, so a CatBoost correlation below that is genuine cross-family")
    print("disagreement and the thresholds in 06 can be read as written.")

06_catboost.ipynb thresholds: alive below 0.97, dead above 0.99

tightest within-family pair (lr only) : 0.9981
loosest within-family pair (lr only)  : 0.9882
loosest pair of any kind (capacity)   : 0.9741

VERDICT ON THE THRESHOLDS: the 0.99 'same model' bar is unsafe. Same-family
runs already sit below it, so it cannot separate families. The 0.97 bar
still discriminates. Treat 0.99 as uninformative rather than as a kill.


## Question 2: the blending floor

Rank average, not probability average, for the reason in `NOTES.md`: AUC reads only
ordering, so rank averaging is immune to models sitting on different probability
scales.

Both fold-mean and pooled AUC are reported. The bar for believing any of this is the
fold standard deviation of the best single model, which the deterministic runs put near
0.00055.

In [6]:
from itertools import combinations

BAR = float(recon.loc[recon["key"] == "exp8", "fold_sd"].iloc[0])
best_key = recon.loc[recon["fold_mean"].idxmax(), "key"]
best_fm = float(recon["fold_mean"].max())


def blend(ks, weights=None):
    w = np.ones(len(ks)) / len(ks) if weights is None else np.asarray(weights, float)
    w = w / w.sum()
    return sum(wi * ranks[k] for wi, k in zip(w, ks))


rows = []
for r in (2, 3):
    for combo in combinations(det_keys, r):
        b = blend(list(combo))
        rows.append({"blend": " + ".join(combo), "n": r,
                     "fold_mean": fold_mean_auc(b), "fold_sd": fold_sd_auc(b),
                     "pooled": float(roc_auc_score(y, b))})

# All seven, including the non-deterministic era, as the widest available spread.
b_all = blend(keys)
rows.append({"blend": f"all {len(keys)} (noisy era included)", "n": len(keys),
             "fold_mean": fold_mean_auc(b_all), "fold_sd": fold_sd_auc(b_all),
             "pooled": float(roc_auc_score(y, b_all))})

bl = pd.DataFrame(rows)
bl["vs_best"] = bl["fold_mean"] - best_fm
bl["sd_units"] = bl["vs_best"] / BAR
bl = bl.sort_values("fold_mean", ascending=False)

print(f"best single model: {best_key} at fold-mean {best_fm:.6f}")
print(f"bar for belief   : {BAR:.6f} (fold sd of exp8)\n")
print(bl.to_string(index=False,
                   formatters={"fold_mean": "{:.6f}".format,
                               "fold_sd": "{:.6f}".format,
                               "pooled": "{:.6f}".format,
                               "vs_best": "{:+.6f}".format,
                               "sd_units": "{:+.2f}".format}))

best single model: exp8 at fold-mean 0.963275
bar for belief   : 0.000549 (fold sd of exp8)

                     blend  n fold_mean  fold_sd   pooled   vs_best sd_units
               exp7 + exp8  2  0.963347 0.000565 0.963346 +0.000072    +0.13
        exp6 + exp7 + exp8  3  0.963315 0.000575 0.963314 +0.000040    +0.07
               exp6 + exp8  2  0.963201 0.000579 0.963200 -0.000073    -0.13
               exp6 + exp7  2  0.963192 0.000601 0.963191 -0.000083    -0.15
all 7 (noisy era included)  7  0.962380 0.000668 0.962379 -0.000895    -1.63


In [7]:
top = bl.iloc[0]
print(f"best blend: {top['blend']}")
print(f"gain over the best single model: {top['vs_best']:+.6f} "
      f"({top['sd_units']:+.2f} fold sd)\n")

if top["vs_best"] <= 0:
    print("FLOOR: zero. Blending within one family buys nothing at all here, so any")
    print("gain a diverse model produces is entirely attributable to the diversity.")
elif top["sd_units"] < 1:
    print("FLOOR: positive but inside noise. A cross-family blend has to clear roughly")
    print("one fold sd on top of this to count as a real contribution.")
else:
    print("FLOOR: material. Rank-averaging near-identical models already clears the")
    print("bar, which means part of any future blend gain is just averaging and not")
    print("diversity. Subtract this floor before crediting CatBoost with anything.")

print("\nAlso worth reading: the fold_sd column. Blending is a variance-reduction")
print("device as much as a score device, and a lower fold sd is what survives a")
print("resample onto the private split even when the mean barely moves.")

best blend: exp7 + exp8
gain over the best single model: +0.000072 (+0.13 fold sd)

FLOOR: positive but inside noise. A cross-family blend has to clear roughly
one fold sd on top of this to count as a real contribution.

Also worth reading: the fold_sd column. Blending is a variance-reduction
device as much as a score device, and a lower fold sd is what survives a
resample onto the private split even when the mean barely moves.


## What this changes

Run top to bottom on a clean kernel, 2026-08-04. No ledger row: no model was trained
and no submission was produced.

**1. Provenance holds.** All seven saved vectors reproduce their `experiments.csv`
fold-mean to within 4.6e-7. Under the notebook layout that reconciliation is the only
provenance a ledger row has, and it passes.

**2. The correlation thresholds in `06_catboost.ipynb` are wrong, and one of them is
dangerous.** Within-family Spearman spans **0.9741 to 0.9981** across seven runs that
differ only in tree count and learning rate. The 0.99 "these are the same model wearing
different hats" bar sits *inside* the range a single family produces on its own, so it
cannot separate a family from itself and must never be read as a kill. The 0.97 bar is
below anything the family produces, so it discriminates, but a second GBDT is unlikely
to reach it.

**3. The finding that actually changes the plan: less correlation produced worse blends
here, not better ones.**

| blend | Spearman of the pair | vs best single |
|---|---|---|
| exp7 + exp8 | 0.9981, the most correlated pair | **+0.000072** |
| exp6 + exp8 | 0.9882 | -0.000073 |
| exp6 + exp7 | 0.9892 | -0.000083 |
| all seven | down to 0.9741 | -0.000895 |

The ranking is monotone in the wrong direction, and the cause is mundane. The
low-correlation pairs are low-correlation because one member is *worse*, underfit at 100
trees or past the turnover at 2000. Their disagreement comes from one model being wrong
more often rather than from complementary errors, and averaging that in costs more than
the decorrelation returns.

**So Spearman is the wrong decision statistic and `06_catboost.ipynb` should not gate on
it.** Correlation is a proxy for "will blending help", and on the only data available to
test that proxy, it was anti-predictive. The measured blend AUC is available directly and
costs nothing extra to read. Gate on that. Correlation stays in the notebook as description.
The verdict comes from the blend.

**4. The blending floor is +0.000072, or 0.13 fold sd.** Inside noise. A cross-family
blend has to clear roughly 0.00055 on top of the best single model before any of it
counts.

**5. No blend reduced fold spread.** Best single sits at 0.000549 and every blend came
out at or above 0.000565. Worth recording because the private-leaderboard case for
ensembling is variance reduction as much as score, and within-family averaging did not
deliver that either.

**What this does not say.** It does not say ensembling is dead here. Every pair tested
varies *capacity* within one algorithm. CatBoost's ordered target statistics on the three
categoricals is a different mechanism rather than a different capacity, and this
diagnostic cannot speak to it. The top of the public leaderboard is still a 55-model stack near
0.9707. What it does say is that the CatBoost probe should be cheap, and should be read
on the blend rather than on the correlation.